# Phase 0: setup, data audit, baseline check

Project: trustworthy mosquito identification on unseen domains (BioDCASE 2026 CD-MSC). See problem-statement.md.

**What this notebook does**
1. Mounts Drive (optional), clones the official baseline repo, installs light dependencies.
2. Lists and downloads the development dataset (Zenodo record 19095788) inside the Colab runtime.
3. Audits it: species x domain counts, sample rate and duration per domain, which test clips are unseen.
4. Evaluates the released seed-42 baseline checkpoint to reproduce BA_unseen (about 0.18).

**Paste back to Claude:** the output of cells 3, 5, 6 and 7 (file list, species x domain table, per-domain audio table, baseline metrics).

**Gate:** the baseline reproduces BA_unseen of roughly 0.17 to 0.19. If it does not, stop and report the numbers.

Run the cells top to bottom. If a cell fails, paste the error back.

In [ ]:
# CELL 1: config and helpers
import os, sys, json, subprocess, time, hashlib, glob, zipfile, random

USE_DRIVE = True          # True: keep data and features on Google Drive so they survive runtime resets
RUN_BASELINE_EVAL = True  # extract features and evaluate the released seed-42 checkpoint on the dev test split

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    ROOT = '/content/drive/MyDrive/mosquito_cdmsc'
else:
    ROOT = '/content/mosquito_cdmsc'
os.makedirs(ROOT, exist_ok=True)
REPO = ROOT + '/CD-MSC'
DL = ROOT + '/downloads'
os.makedirs(DL, exist_ok=True)

def sh(cmd, cwd=None, tail=40):
    p = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    out = (p.stdout or '') + (p.stderr or '')
    print('\n'.join(out.splitlines()[-tail:]))
    if p.returncode != 0:
        raise RuntimeError('command failed: ' + ' '.join(cmd))
    return out

def find_src():
    pats = [ROOT + '/extracted/raw_audio', ROOT + '/extracted/*/raw_audio', ROOT + '/extracted/*/*/raw_audio',
            DL + '/raw_audio', DL + '/*/raw_audio', DL + '/*/*/raw_audio']
    for p in pats:
        hit = glob.glob(p)
        if hit:
            return os.path.dirname(hit[0])
    return None

print('ROOT =', ROOT)
try:
    sh(['nvidia-smi'], tail=12)
except Exception as e:
    print('no GPU visible (fine for this notebook):', e)

In [ ]:
# CELL 2: clone the official baseline and install light dependencies
# (the repo pins torch and numpy versions; we skip those pins so Colab's own versions are kept)
if not os.path.exists(REPO + '/train.py'):
    sh(['git', 'clone', '--depth', '1', 'https://github.com/Yuanbo2020/CD-MSC', REPO])
sh([sys.executable, '-m', 'pip', '-q', 'install', 'librosa==0.11.0', 'torchlibrosa==0.1.0', 'soundfile', 'requests', 'pandas'])
print(sorted(os.listdir(REPO)))
print('released checkpoints in clone:', len(glob.glob(REPO + '/outputs/MTRCNN_seed*/model/model_best.pth')))

In [ ]:
# CELL 3: list the files in the development-dataset Zenodo record (PASTE THIS OUTPUT BACK)
import requests
REC = '19095788'
r = requests.get('https://zenodo.org/api/records/' + REC, timeout=60)
r.raise_for_status()
j = r.json()
files = j.get('files')
if files is None:
    files = requests.get('https://zenodo.org/api/records/' + REC + '/files', timeout=60).json()['entries']
print('TITLE:', j['metadata']['title'])
total = 0
for f in files:
    total += f['size']
    print(f['key'], '|', round(f['size'] / 1e6, 1), 'MB |', f.get('checksum', ''))
print('TOTAL GB:', round(total / 1e9, 2))

In [ ]:
# CELL 4: download (resumable), verify checksums, extract zips
def md5(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, 'rb') as fh:
        for b in iter(lambda: fh.read(chunk), b''):
            h.update(b)
    return h.hexdigest()

for f in files:
    dst = os.path.join(DL, f['key'])
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    url = (f.get('links') or {}).get('self') or ('https://zenodo.org/records/' + REC + '/files/' + f['key'] + '?download=1')
    if not (os.path.exists(dst) and os.path.getsize(dst) == f['size']):
        sh(['wget', '-c', '-q', '-O', dst, url], tail=5)
    want = f.get('checksum', '').replace('md5:', '')
    ok = (not want) or md5(dst) == want
    print(f['key'], 'OK' if ok else 'CHECKSUM MISMATCH')
    if dst.endswith('.zip') and ok and find_src() is None:
        with zipfile.ZipFile(dst) as z:
            z.extractall(ROOT + '/extracted')

SRC = find_src()
assert SRC, 'raw_audio folder not found - paste the file list from cell 3 back to Claude'
print('SRC =', SRC)
print(sorted(os.listdir(SRC)))

In [ ]:
# CELL 5: audit - species x domain counts, split, unseen test clips, audio properties (PASTE THIS OUTPUT BACK)
import numpy as np, pandas as pd, soundfile as sf
SRC = globals().get('SRC') or find_src()
RAW = SRC + '/raw_audio'
META = SRC + '/metadata'
SP = {1: 'Ae.aeg', 2: 'Ae.alb', 3: 'Cx.qui', 4: 'An.gam', 5: 'An.ara', 6: 'An.dir', 7: 'Cx.pip', 8: 'An.min', 9: 'An.ste'}

names = [os.path.splitext(x)[0] for x in os.listdir(RAW) if x.endswith('.wav')]
df = pd.DataFrame({'id': names})
parts = df['id'].str.split('_')
df['species'] = parts.str[1].astype(int)
df['domain'] = parts.str[3].astype(int)
print(len(df), 'clips (README says 271380)')
tab = pd.crosstab(df['species'].map(SP), 'D' + df['domain'].astype(str), margins=True)
print('\nCLIPS PER SPECIES x DOMAIN\n', tab)

def read_ids(name):
    p = META + '/' + name + '_ids.txt'
    return [os.path.splitext(l.strip())[0] for l in open(p) if l.strip()]
split_of = {}
for name in ['Training', 'Validation', 'Test']:
    for i in read_ids(name):
        split_of[i] = name
df['split'] = df['id'].map(split_of)
print('\nSPLIT SIZES\n', df['split'].value_counts(dropna=False))

tv = df[df['split'].isin(['Training', 'Validation'])]
train_pairs = set(zip(tv['species'], tv['domain']))
te = df[df['split'] == 'Test'].copy()
te['seen'] = [(s, d) in train_pairs for s, d in zip(te['species'], te['domain'])]
print('\nTEST CLIPS: seen vs unseen (species, domain) pair, per species\n',
      pd.crosstab(te['species'].map(SP), te['seen'].map({True: 'seen', False: 'unseen'}), margins=True))
print('\nTEST UNSEEN CLIPS PER DOMAIN\n', te[~te['seen']].groupby('domain').size())
sp_json = META + '/split_summary.json'
if os.path.exists(sp_json):
    print('\nsplit_summary.json (first 1500 chars):\n', open(sp_json).read()[:1500])

rows = []
for d, g in df.groupby('domain'):
    pick = g.sample(min(150, len(g)), random_state=0)
    for i in pick['id']:
        info = sf.info(RAW + '/' + i + '.wav')
        rows.append((d, info.samplerate, info.duration, info.channels))
au = pd.DataFrame(rows, columns=['domain', 'sr', 'dur', 'ch'])
audio_tab = au.groupby('domain').agg(n=('sr', 'size'), sample_rates=('sr', lambda s: sorted(set(s))),
                                      dur_mean=('dur', 'mean'), dur_min=('dur', 'min'), dur_max=('dur', 'max'),
                                      channels=('ch', lambda s: sorted(set(s)))).round(2)
print('\nAUDIO PROPERTIES PER DOMAIN (sample of up to 150 clips each)\n', audio_tab)

os.makedirs(ROOT + '/audit', exist_ok=True)
tab.to_csv(ROOT + '/audit/species_by_domain.csv')
audio_tab.to_csv(ROOT + '/audit/audio_by_domain.csv')
df.to_csv(ROOT + '/audit/all_clips_with_split.csv', index=False)

In [ ]:
# CELL 6: reproduce the baseline - extract features, evaluate the released seed-42 checkpoint (PASTE THE METRICS BACK)
# Feature extraction covers all 271k clips and can take a while; it is skipped if the features already exist.
SRC = globals().get('SRC') or find_src()
RAW = SRC + '/raw_audio'
META = SRC + '/metadata'
cfg = json.load(open(REPO + '/configs/default_experiment.json'))
cfg.update(dataset_root=RAW, train_ids_path=META + '/Training_ids.txt', val_ids_path=META + '/Validation_ids.txt',
           test_ids_path=META + '/Test_ids.txt', feature_root=ROOT + '/features', output_root=ROOT + '/outputs_repro')
CFG = REPO + '/configs/colab_experiment.json'
json.dump(cfg, open(CFG, 'w'), indent=2)
print('wrote', CFG)

if RUN_BASELINE_EVAL:
    if not os.path.exists(cfg['feature_root'] + '/test_features.pkl'):
        t0 = time.time()
        sh([sys.executable, 'extract_features.py', '--config', CFG], cwd=REPO)
        print('feature extraction minutes:', round((time.time() - t0) / 60, 1))
    ckpt = REPO + '/outputs/MTRCNN_seed42_B64_E100_earlystop_min10_pati5/model/model_best.pth'
    print('released checkpoint present:', os.path.exists(ckpt))
    if os.path.exists(ckpt):
        out = ROOT + '/outputs_repro/seed42_test_metrics.json'
        os.makedirs(os.path.dirname(out), exist_ok=True)
        sh([sys.executable, 'evaluate.py', '--config', CFG, '--checkpoint', ckpt, '--split', 'test', '--metrics-out', out], cwd=REPO)
        print(open(out).read()[:3000])
    else:
        print('Checkpoint not in the clone. Tell Claude; the fallback is to train one seed with train.py.')

In [ ]:
# CELL 7: one-screen summary to paste back
summary = {
    'zenodo_files': [(f['key'], f['size']) for f in files],
    'n_clips': int(len(df)),
    'split_sizes': {k: int(v) for k, v in df['split'].value_counts().items()},
    'test_unseen_clips': int((~te['seen']).sum()),
    'test_seen_clips': int(te['seen'].sum()),
    'domain_sample_rates': {int(d): sorted(set(int(x) for x in g['sr'])) for d, g in au.groupby('domain')},
    'domain_mean_duration_s': {int(d): round(float(g['dur'].mean()), 2) for d, g in au.groupby('domain')},
}
mp = ROOT + '/outputs_repro/seed42_test_metrics.json'
if os.path.exists(mp):
    summary['baseline_seed42_test_metrics'] = json.load(open(mp))
json.dump(summary, open(ROOT + '/audit/phase0_summary.json', 'w'), indent=2)
print(json.dumps(summary, indent=2)[:4000])